In [1]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
import yaml


Set parameter TokenServer to value "sophia1.hpc.ait.dtu.dk"


**Set Up**

In [2]:
fn = 'resources/Nordics20_test/networks/base_s_20__12h_2050.nc'


In [3]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.nordics20.yaml").read_text())


INFO:pypsa.network.io:New version 1.1.0 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


In [4]:
p = Path(fn)  
try:
   if p.exists():
       p.unlink()
       print(f"Deleted {p}")
   else:
       print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

Deleted resources/Nordics20_test/networks/base_s_20__12h_2050.nc


**Options**

In [5]:
ongrid=False
cluster_cost_reduction=0
cluster_size=1000   
renewables={"solar",'solar-hsat','onwind'}

In [6]:
nodes_with_clusters = n.buses.loc[
    n.buses.index.str[:2].isin(config['countries']) &
    (n.buses['carrier'] == 'AC')
].index.tolist()




In [7]:
nodes_with_clusters

['DE0 0',
 'DE0 1',
 'DE0 2',
 'DE0 3',
 'DE0 4',
 'DE0 5',
 'DE0 6',
 'DE0 7',
 'DK0 0',
 'DK1 0',
 'GB2 0',
 'GB2 1',
 'GB2 2',
 'GB2 3',
 'GB2 4',
 'GB3 0',
 'NL0 0',
 'NO1 0',
 'SE1 0',
 'SE1 1']

**Buses and Generators of the Cluster Addition**

In [8]:
def assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables):
    
    nodes_renewables_cf = {}                #dictionary of dataframes by node and renewable type, sorting the generators by average capacity factor (descending order)
    clusters_generators={}                      #dictionary of dataframes by node and renewable type, containing the generators assigned to the cluster  

    for node in nodes_with_clusters:
        for renewable in renewables:

            nodes_renewables_cf[(node, renewable)] = pd.DataFrame(
                index=n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")].index,
                columns=["p_max_pu","p_nom_max"]  
            )
            print(nodes_renewables_cf[(node, renewable)])

            clusters_generators[(node, renewable)] = pd.DataFrame()

            #we are considering the highest mean p_min_pu to determine the best generators per renewable available

            nodes_renewables_cf[(node, renewable)] ["p_max_pu"] = n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.astype(str).str.contains(rf"{node}.*{renewable}$")].mean()
            nodes_renewables_cf[(node, renewable)] ["p_nom_max"] = n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")]

            nodes_renewables_cf[(node, renewable)] = nodes_renewables_cf[(node, renewable)].sort_values("p_max_pu", ascending=False)

            print(nodes_renewables_cf[(node, renewable)])

            #print(nodes_renewables_cf[(country, renewable)])

            number_gen=0



            while  nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() <= cluster_size:

                if number_gen > len(nodes_renewables_cf[(node, renewable)]):
                    raise ValueError(f"Not enough {renewable} generators to reach cluster_size.")
                
                number_gen=number_gen+1

            #print(f"{renewable} generators in cluster: {number_gen+1}")

            clusters_generators[(node, renewable)]  = n.generators.loc[nodes_renewables_cf[(node, renewable)].index[0:number_gen+1]]
            remaining_capacity = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size
            #nodes_renewables_cf[(country, renewable)].iloc[number_gen]["p_nom_max"] = remaining_capacity maybe it is better to do this step later

            print(f"Remaining top {renewable} capacity outside the cluster: {remaining_capacity} MW")

            
            clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], "p_nom_max"] = cluster_size - clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[0:number_gen],"p_nom_max"].sum()

            print(f"Capacity of the last {renewable} generator adjusted to fit cluster size: {clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], 'p_nom_max']} MW")

            print(clusters_generators[(node, renewable)])

            for idx in clusters_generators[(node, renewable)].index:

                ### Electricity bus and generators ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' cluster'}$").any():
        
                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                        v_nom=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "v_nom"],
                        x=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "x"],
                        y=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "y"],
                        unit=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "unit"],
                        location=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "location"],
                        country=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "country"],
                        carrier=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "carrier"],
                        control=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "control"],
                        substation_lv=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_lv"],
                        substation_off=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_off"],
                    )

                n.add(
                    "Generator",
                    name=clusters_generators[(node, renewable)].loc[idx].name + " cluster",
                    bus=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                    carrier=clusters_generators[(node, renewable)].loc[idx].carrier,
                    p_nom_max=clusters_generators[(node, renewable)].loc[idx].p_nom_max,
                    p_max_pu=clusters_generators[(node, renewable)].loc[idx].p_max_pu,
                    marginal_cost=clusters_generators[(node, renewable)].loc[idx].marginal_cost*(1-cluster_cost_reduction),
                    capital_cost=clusters_generators[(node, renewable)].loc[idx].capital_cost*(1-cluster_cost_reduction),
                    efficiency=clusters_generators[(node, renewable)].loc[idx].efficiency,
                    location=clusters_generators[(node, renewable)].loc[idx].location,
                    unit=clusters_generators[(node, renewable)].loc[idx].unit,
                    p_nom_extendable=True,
                    overwrite=True,)


                
                
                n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name + " cluster"] = n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name]


                ### H2 bus ##

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2 cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " H2 cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node,renewable)].loc[idx].bus + ' H2'}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "substation_off"],
                    )

                ### methanol bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' methanol cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " methanol cluster",
                        v_nom=n.buses.at["EU methanol", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus }", "y"],
                        unit=n.buses.at["EU methanol", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus  }", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus }", "country"],
                        carrier=n.buses.at["EU methanol", "carrier"],
                        control=n.buses.at["EU methanol", "control"],
                        substation_lv=n.buses.at["EU methanol", "substation_lv"],
                        substation_off=n.buses.at["EU methanol", "substation_off"],
                    )
                
                ### Batteries bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " battery cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "substation_off"],
                    )



                if idx == nodes_renewables_cf[(node, renewable)].iloc[number_gen].name:
                    n.generators.loc[n.generators.index == idx, "p_nom_max"] = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size

                    print(f"Residual capacity of generator {clusters_generators[(node, renewable)].loc[idx].name} is {n.generators.loc[n.generators.index == idx, 'p_nom_max']} MW")
                
                else:


                    n.remove(
                            "Generator",
                            name=clusters_generators[(node, renewable)].loc[idx].name,
                    )



            

    return n

n= assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables)
            



            

            





        




              p_max_pu p_nom_max
name                            
DE0 0 0 solar      NaN       NaN
DE0 0 1 solar      NaN       NaN
DE0 0 2 solar      NaN       NaN
DE0 0 3 solar      NaN       NaN
DE0 0 4 solar      NaN       NaN
               p_max_pu     p_nom_max
name                                 
DE0 0 4 solar  0.115972   1213.854299
DE0 0 3 solar  0.113219  35881.757979
DE0 0 2 solar  0.109159  73087.653197
DE0 0 1 solar  0.106340  51059.805587
DE0 0 0 solar  0.103268   5183.277072
Remaining top solar capacity outside the cluster: 213.8542994490972 MW
Capacity of the last solar generator adjusted to fit cluster size: 1000.0 MW
                 bus control type      p_nom  p_nom_mod  p_nom_extendable  \
name                                                                        
DE0 0 4 solar  DE0 0      PQ       30.153724        0.0              True   

               p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  up_time_before  \
name                                      

Residual capacity of generator DE0 1 4 solar is name
DE0 1 4 solar    784.005757
Name: p_nom_max, dtype: float64 MW
                   p_max_pu p_nom_max
name                                 
DE0 1 0 solar-hsat      NaN       NaN
                    p_max_pu     p_nom_max
name                                      
DE0 1 0 solar-hsat  0.134574  72823.376849
Remaining top solar-hsat capacity outside the cluster: 71823.37684905133 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
DE0 1 0 solar-hsat  DE0 1      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  \
name                                                           ...   
DE0 1 0 solar-hsat        0.0     1000.0        NaN       0.0  ...   

                    up_time_before  d

Residual capacity of generator DE0 2 4 onwind is name
DE0 2 4 onwind    15187.656743
Name: p_nom_max, dtype: float64 MW
              p_max_pu p_nom_max
name                            
DE0 3 0 solar      NaN       NaN
DE0 3 1 solar      NaN       NaN
DE0 3 2 solar      NaN       NaN
DE0 3 3 solar      NaN       NaN
DE0 3 4 solar      NaN       NaN
               p_max_pu     p_nom_max
name                                 
DE0 3 4 solar  0.108534   2826.876412
DE0 3 3 solar  0.106685  66770.399793
DE0 3 2 solar  0.104919  75854.129485
DE0 3 1 solar  0.103366  15211.396088
DE0 3 0 solar  0.100753    637.528032
Remaining top solar capacity outside the cluster: 1826.876411837719 MW
Capacity of the last solar generator adjusted to fit cluster size: 1000.0 MW
                 bus control type       p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
DE0 3 4 solar  DE0 3      PQ       101.168221        0.0              True   



Residual capacity of generator DE0 3 4 solar is name
DE0 3 4 solar    1826.876412
Name: p_nom_max, dtype: float64 MW
                   p_max_pu p_nom_max
name                                 
DE0 3 0 solar-hsat      NaN       NaN
                    p_max_pu      p_nom_max
name                                       
DE0 3 0 solar-hsat  0.125266  140109.894325
Remaining top solar-hsat capacity outside the cluster: 139109.89432519046 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
DE0 3 0 solar-hsat  DE0 3      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  \
name                                                           ...   
DE0 3 0 solar-hsat        0.0     1000.0        NaN       0.0  ...   

                    up_time_befo

Residual capacity of generator DE0 5 3 solar is name
DE0 5 3 solar    10717.822665
Name: p_nom_max, dtype: float64 MW
                   p_max_pu p_nom_max
name                                 
DE0 5 0 solar-hsat      NaN       NaN
                    p_max_pu      p_nom_max
name                                       
DE0 5 0 solar-hsat  0.129106  172434.235771
Remaining top solar-hsat capacity outside the cluster: 171434.23577122472 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
DE0 5 0 solar-hsat  DE0 5      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  \
name                                                           ...   
DE0 5 0 solar-hsat        0.0     1000.0        NaN       0.0  ...   

                    up_time_bef

...

Residual capacity of generator GB2 2 4 onwind is name
GB2 2 4 onwind    3627.290662
Name: p_nom_max, dtype: float64 MW
              p_max_pu p_nom_max
name                            
GB2 3 0 solar      NaN       NaN
GB2 3 1 solar      NaN       NaN
GB2 3 2 solar      NaN       NaN
GB2 3 3 solar      NaN       NaN
GB2 3 4 solar      NaN       NaN
               p_max_pu     p_nom_max
name                                 
GB2 3 4 solar  0.119390  14344.421578
GB2 3 3 solar  0.115547  40399.880824
GB2 3 2 solar  0.110973  72693.337664
GB2 3 1 solar  0.107177  16643.770168
GB2 3 0 solar  0.102294  15574.519970
Remaining top solar capacity outside the cluster: 13344.421577932218 MW
Capacity of the last solar generator adjusted to fit cluster size: 1000.0 MW
                 bus control type       p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
GB2 3 4 solar  GB2 3      PQ       558.412606        0.0              True   



Residual capacity of generator GB2 4 0 solar-hsat is name
GB2 4 0 solar-hsat    122752.163624
Name: p_nom_max, dtype: float64 MW
               p_max_pu p_nom_max
name                             
GB2 4 0 onwind      NaN       NaN
GB2 4 1 onwind      NaN       NaN
GB2 4 2 onwind      NaN       NaN
GB2 4 3 onwind      NaN       NaN
GB2 4 4 onwind      NaN       NaN
                p_max_pu     p_nom_max
name                                  
GB2 4 4 onwind  0.518920   1422.202945
GB2 4 3 onwind  0.473176   1538.934914
GB2 4 2 onwind  0.426901   1972.492555
GB2 4 1 onwind  0.369406  19748.582022
GB2 4 0 onwind  0.318590  38673.293014
Remaining top onwind capacity outside the cluster: 422.2029447468185 MW
Capacity of the last onwind generator adjusted to fit cluster size: 1000.0 MW
                  bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                     
GB2 4 4 onwind  GB2 4      PQ        45.0        0.0          

Residual capacity of generator NL0 0 4 solar is name
NL0 0 4 solar    4978.350652
Name: p_nom_max, dtype: float64 MW
                   p_max_pu p_nom_max
name                                 
NL0 0 0 solar-hsat      NaN       NaN
                    p_max_pu      p_nom_max
name                                       
NL0 0 0 solar-hsat  0.129457  121237.554108
Remaining top solar-hsat capacity outside the cluster: 120237.55410777082 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
NL0 0 0 solar-hsat  NL0 0      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  \
name                                                           ...   
NL0 0 0 solar-hsat        0.0     1000.0        NaN       0.0  ...   

                    up_time_befo

Residual capacity of generator NO1 0 4 onwind is name
NO1 0 4 onwind    706.763199
Name: p_nom_max, dtype: float64 MW
              p_max_pu p_nom_max
name                            
SE1 0 0 solar      NaN       NaN
SE1 0 1 solar      NaN       NaN
SE1 0 2 solar      NaN       NaN
SE1 0 3 solar      NaN       NaN
SE1 0 4 solar      NaN       NaN
               p_max_pu     p_nom_max
name                                 
SE1 0 4 solar  0.123355   3761.705803
SE1 0 3 solar  0.118497  39869.849254
SE1 0 2 solar  0.114353  70388.876402
SE1 0 1 solar  0.109405  25559.470120
SE1 0 0 solar  0.106147   2618.063829
Remaining top solar capacity outside the cluster: 2761.7058025973256 MW
Capacity of the last solar generator adjusted to fit cluster size: 1000.0 MW
                 bus control type      p_nom  p_nom_mod  p_nom_extendable  \
name                                                                        
SE1 0 4 solar  SE1 0      PQ       30.454336        0.0              True   

    

Residual capacity of generator SE1 1 0 solar-hsat is name
SE1 1 0 solar-hsat    78668.555294
Name: p_nom_max, dtype: float64 MW
               p_max_pu p_nom_max
name                             
SE1 1 0 onwind      NaN       NaN
SE1 1 1 onwind      NaN       NaN
SE1 1 2 onwind      NaN       NaN
SE1 1 3 onwind      NaN       NaN
SE1 1 4 onwind      NaN       NaN
                p_max_pu      p_nom_max
name                                   
SE1 1 4 onwind  0.413929    2662.999080
SE1 1 3 onwind  0.339047   14243.793414
SE1 1 2 onwind  0.250521   56286.884012
SE1 1 1 onwind  0.187823  487708.055361
SE1 1 0 onwind  0.103959   45211.830477
Remaining top onwind capacity outside the cluster: 1662.9990802362072 MW
Capacity of the last onwind generator adjusted to fit cluster size: 1000.0 MW
                  bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                     
SE1 1 4 onwind  SE1 1      PQ        10.0        0.0   

In [9]:
n.buses.loc[n.buses.index.str.contains("methanol cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
name,,,,,,,,,,,,,,,,
DE0 0 methanol cluster,1.0,,13.458377,52.417805,methanol,MWh_th,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 1 methanol cluster,1.0,,8.293987,49.606082,methanol,MWh_th,DE0 1,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 2 methanol cluster,1.0,,11.879841,48.480859,methanol,MWh_th,DE0 2,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 3 methanol cluster,1.0,,8.434069,52.475538,methanol,MWh_th,DE0 3,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 4 methanol cluster,1.0,,7.080230,51.223591,methanol,MWh_th,DE0 4,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 5 methanol cluster,1.0,,10.265232,53.405130,methanol,MWh_th,DE0 5,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 6 methanol cluster,1.0,,9.471163,48.681443,methanol,MWh_th,DE0 6,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 7 methanol cluster,1.0,,11.061228,50.508897,methanol,MWh_th,DE0 7,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DK0 0 methanol cluster,1.0,,9.727929,56.142471,methanol,MWh_th,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN


In [10]:
n.generators_t['p_max_pu']

name,DE0 0 0 offwind-ac,DE0 0 0 offwind-float,DE0 0 0 onwind,DE0 0 0 solar,DE0 0 0 solar rooftop,DE0 0 0 solar-hsat,DE0 0 1 onwind,DE0 0 1 solar,DE0 0 1 solar rooftop,DE0 0 2 onwind,...,NL0 0 3 onwind cluster,NO1 0 4 solar cluster,NO1 0 0 solar-hsat cluster,NO1 0 4 onwind cluster,SE1 0 4 solar cluster,SE1 0 0 solar-hsat cluster,SE1 0 4 onwind cluster,SE1 1 4 solar cluster,SE1 1 0 solar-hsat cluster,SE1 1 4 onwind cluster
snapshot,,,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.885500,0.865382,0.668417,0.056416,0.056416,0.041229,0.767919,0.056738,0.056738,0.941190,...,0.577368,0.013872,0.009266,0.617372,0.007216,0.027491,0.992861,0.026330,0.019140,0.434222
2013-01-01 12:00:00,0.687291,0.465413,0.487492,0.015802,0.015802,0.012184,0.450173,0.019273,0.019273,0.421511,...,0.851731,0.007533,0.004823,0.403871,0.002338,0.008017,0.541646,0.003967,0.003081,0.179009
2013-01-02 00:00:00,0.877850,0.703120,0.478048,0.049793,0.049793,0.055901,0.537082,0.063763,0.063763,0.687848,...,0.743640,0.012977,0.008648,0.238663,0.021945,0.031705,0.666617,0.020717,0.013949,0.185520
2013-01-02 12:00:00,0.885042,0.788131,0.509659,0.015406,0.015406,0.016345,0.567024,0.018221,0.018221,0.772742,...,0.799629,0.006878,0.004207,0.408567,0.004903,0.014160,0.782770,0.003123,0.002344,0.200599
2013-01-03 00:00:00,0.885494,0.854945,0.789928,0.008803,0.008803,0.016241,0.834121,0.015879,0.015879,0.934728,...,0.973061,0.009181,0.005952,0.688372,0.010255,0.011225,0.834281,0.007675,0.005723,0.091640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-29 12:00:00,0.871631,0.705271,0.371346,0.037088,0.037088,0.020665,0.456289,0.029842,0.029842,0.651600,...,0.886826,0.004667,0.001965,0.161766,0.005210,0.007175,0.900232,0.006373,0.004764,0.824024
2013-12-30 00:00:00,0.864618,0.691656,0.238286,0.158278,0.158278,0.119247,0.345616,0.151625,0.151625,0.625659,...,0.975430,0.007159,0.004548,0.487115,0.069811,0.033784,0.967619,0.055146,0.037400,0.636971
2013-12-30 12:00:00,0.845183,0.669323,0.205363,0.077168,0.077168,0.061177,0.375544,0.073384,0.073384,0.651527,...,0.998983,0.002431,0.001642,0.981489,0.016077,0.011450,0.872472,0.008453,0.006359,0.538245


In [11]:
n.generators.loc[n.generators.index.str.contains('cluster')]

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt,location,unit
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 4 solar cluster,DE0 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 0 solar-hsat cluster,DE0 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 4 onwind cluster,DE0 0 cluster,PQ,,0.0,0.0,True,0.0,490.848899,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 3 onwind cluster,DE0 0 cluster,PQ,,0.0,0.0,True,0.0,509.151101,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 1 4 solar cluster,DE0 1 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 0 0 solar-hsat cluster,SE1 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
SE1 0 4 onwind cluster,SE1 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
SE1 1 4 solar cluster,SE1 1 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,


**Links of the Cluster Addition**

In [12]:
def add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid):

    for node in nodes_with_clusters:

        ### H2 Electrolysis ###

        link_name = f"{node} H2 Electrolysis"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"] + " cluster",
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        ### Methanolization ###

        ### Methanolization ###

        link_name = f"{node} methanolisation"
        cluster_methanol_bus_name = f"{node} methanol cluster"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=cluster_methanol_bus_name,
            bus2=n.links.at[link_name, "bus2"] + " cluster",
            bus3=n.links.at[link_name, "bus3"],
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        n.add(
            "Link",
            name=f"{node} methanol cluster",
            bus0=cluster_methanol_bus_name,
            bus1=n.links.at[link_name, "bus1"],
            p_nom_extendable=True,
            carrier=n.buses.loc[n.links.at[link_name, "bus1"], "carrier"],  
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,


        )

    if ongrid==True :

        ### Electricity connection to grid ###

        link_name = f"{node} electricity cluster"
        
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node} cluster",
            bus1=f"{node}",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,
        )

        link_name = f"{node} electricity cluster back"
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node}",
            bus1=f"{node} cluster",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=True,
            overwrite=True,
        )

    else:
        if f"{node} cluster electricity" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity",
            )
        if f"{node} cluster electricity back" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity back",
            )

    return n

n = add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid)



        


        

In [13]:
n.links.loc[n.links.index.str.contains("methanolisation cluster")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 methanolisation cluster,DE0 0 H2 cluster,DE0 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 1 methanolisation cluster,DE0 1 H2 cluster,DE0 1 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 2 methanolisation cluster,DE0 2 H2 cluster,DE0 2 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 3 methanolisation cluster,DE0 3 H2 cluster,DE0 3 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 4 methanolisation cluster,DE0 4 H2 cluster,DE0 4 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 5 methanolisation cluster,DE0 5 H2 cluster,DE0 5 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 6 methanolisation cluster,DE0 6 H2 cluster,DE0 6 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 7 methanolisation cluster,DE0 7 H2 cluster,DE0 7 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK0 0 methanolisation cluster,DK0 0 H2 cluster,DK0 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


**Storages of the Cluster Addition**

In [14]:
def add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction):

    for node in nodes_with_clusters:

        link_name = f"{node} H2 Store"

    
        n.add("Store",
            name=link_name + " cluster",
            bus=n.stores.at[link_name, "bus"] + " cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
        
        link_name = f"{node} battery"


        n.add(
                "Link",
                name=link_name + " charger cluster",
                bus0=f"{node} cluster",
                bus1=f"{node} battery cluster",
                carrier=n.buses.at[link_name, "carrier"],   
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=False,
                overwrite=True,
            )
        n.add(
                "Link",
                name=link_name + " discharger cluster",
                bus0=f"{node} battery cluster",
                bus1=f"{node} cluster",
                carrier=n.buses.at[link_name, "carrier"],
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=True,
                overwrite=True,
            )

        n.add("Store",
            name=link_name + " cluster" ,
            bus=f"{node} battery cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
    return n

n = add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction)





In [15]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

In [16]:
n.links.loc[n.links["bus1"].str.contains('methanol')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 solid biomass biomass-to-methanol,DE0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 1 solid biomass biomass-to-methanol,DE0 1 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 2 solid biomass biomass-to-methanol,DE0 2 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 3 solid biomass biomass-to-methanol,DE0 3 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 4 solid biomass biomass-to-methanol,DE0 4 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NO1 0 methanol cluster,NO1 0 methanol cluster,EU methanol,,methanol,1.0000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 0 methanolisation cluster,SE1 0 H2 cluster,SE1 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 0 methanol cluster,SE1 0 methanol cluster,EU methanol,,methanol,1.0000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [17]:
n.links.loc[n.links.index.str.contains("Electrolysis")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Electrolysis,DE0 0,DE0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 1 H2 Electrolysis,DE0 1,DE0 1 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 2 H2 Electrolysis,DE0 2,DE0 2 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 3 H2 Electrolysis,DE0 3,DE0 3 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 4 H2 Electrolysis,DE0 4,DE0 4 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 5 H2 Electrolysis,DE0 5,DE0 5 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 6 H2 Electrolysis,DE0 6,DE0 6 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 7 H2 Electrolysis,DE0 7,DE0 7 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DK0 0 H2 Electrolysis,DK0 0,DK0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0


In [18]:
n.links.loc[n.links["carrier"].str.contains('H2 Electrolysis')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Electrolysis,DE0 0,DE0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 1 H2 Electrolysis,DE0 1,DE0 1 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 2 H2 Electrolysis,DE0 2,DE0 2 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 3 H2 Electrolysis,DE0 3,DE0 3 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 4 H2 Electrolysis,DE0 4,DE0 4 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 5 H2 Electrolysis,DE0 5,DE0 5 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 6 H2 Electrolysis,DE0 6,DE0 6 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 7 H2 Electrolysis,DE0 7,DE0 7 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DK0 0 H2 Electrolysis,DK0 0,DK0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0


In [19]:
n.buses.loc[n.buses.index.str.contains("cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
name,,,,,,,,,,,,,,,,
DE0 0 cluster,380.0,,13.458377,52.417805,AC,MWh_el,DE0 0,1.0,0.0,inf,Slack,,,DE,1.0,1.0
DE0 0 H2 cluster,1.0,,13.458377,52.417805,H2,MWh_LHV,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 0 methanol cluster,1.0,,13.458377,52.417805,methanol,MWh_th,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 0 battery cluster,1.0,,13.458377,52.417805,battery,MWh_el,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 1 cluster,380.0,,8.293987,49.606082,AC,MWh_el,DE0 1,1.0,0.0,inf,PQ,,,DE,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 0 battery cluster,1.0,,13.652177,57.831440,battery,MWh_el,SE1 0,1.0,0.0,inf,PQ,,,SE,NaN,NaN
SE1 1 cluster,380.0,,17.346161,62.213604,AC,MWh_el,SE1 1,1.0,0.0,inf,PQ,,,SE,1.0,1.0
SE1 1 H2 cluster,1.0,,17.346161,62.213604,H2,MWh_LHV,SE1 1,1.0,0.0,inf,PQ,,,SE,NaN,NaN


In [20]:
n.stores.loc[n.stores.index.str.contains("cluster")]



,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_nom_set,e_min_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Store cluster,DE0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,89.430064,0.0,True,0,inf,0.0,NaN
DE0 0 battery cluster,DE0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
DE0 1 H2 Store cluster,DE0 1 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,2224.975719,0.0,True,0,inf,0.0,NaN
DE0 1 battery cluster,DE0 1 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
DE0 2 H2 Store cluster,DE0 2 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,2224.975719,0.0,True,0,inf,0.0,NaN
DE0 2 battery cluster,DE0 2 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
DE0 3 H2 Store cluster,DE0 3 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,89.430064,0.0,True,0,inf,0.0,NaN
DE0 3 battery cluster,DE0 3 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
DE0 4 H2 Store cluster,DE0 4 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,89.430064,0.0,True,0,inf,0.0,NaN


In [21]:
n.links.loc[n.links["carrier"]=='DC']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
relation/13295785-515-DC,NO1 0,GB2 0,,DC,0.955943,True,0,inf,1400.0,0.0,...,0.0,relation/13295785,LINESTRING (-1.5404269162550226 55.14647596191...,1.0,0.983350,,NaN,,False,1068.160241
relation/14126301-450-DC,GB2 1,NL0 0,,DC,0.971354,True,0,inf,1000.0,0.0,...,0.0,relation/14126301,LINESTRING (0.7161575436002887 51.440498299145...,1.0,0.977109,,NaN,,False,380.828296
relation/15775538-600-DC,GB2 2,GB2 4,,DC,0.971389,True,0,inf,2250.0,0.0,...,0.0,relation/15775538,LINESTRING (-4.894821189854914 55.718036768358...,1.0,0.892448,,NaN,,False,379.300168
relation/15781671-525-DC,GB2 0,DK0 0,,DC,0.963113,True,0,inf,1400.0,0.0,...,0.0,relation/15781671,LINESTRING (-0.2365005345670103 52.92099253311...,1.0,0.816407,,NaN,,False,746.986775
relation/16213216-525-DC,DE0 5,NO1 0,,DC,0.959162,True,0,inf,1400.0,0.0,...,0.0,relation/16213216,LINESTRING (6.7544309946114405 58.669060406684...,1.0,0.802530,,NaN,,False,923.664577
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DC42-reversed,DE0 6,DE0 5,,DC,0.968030,True,2037,inf,0.0,0.0,...,1.0,"{""url"":""https://data.netzausbau.de/2037-2023/N...","LINESTRING (10.5331297 53.5252973, 9.0113444 4...",NaN,0.000000,confirmed,NaN,,True,528.166271
DC42plus-reversed,DE0 6,DE0 5,,DC,0.968030,True,2037,inf,0.0,0.0,...,1.0,"{""url"":""https://data.netzausbau.de/2037-2023/N...","LINESTRING (10.5331297 53.5252973, 9.6151453 4...",NaN,0.000000,confirmed,NaN,,True,528.166271
DC5-reversed,DE0 2,DE0 5,,DC,0.967334,True,2027,inf,0.0,0.0,...,1.0,{url:https://www.netzentwicklungsplan.de/sites...,"LINESTRING (11.6267388 52.2484924, 11.5745421 ...",NaN,0.000000,in_permitting,NaN,,True,559.078001


In [22]:
n.stores

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_nom_set,e_min_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
name,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,0.0,0.0,True,0.0,inf,NaN,-1.0,...,0.0,0.0,0.0,0.000000,0.0,True,0,inf,0.0,
DE0 0 co2 stored,DE0 0 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
DE0 1 co2 stored,DE0 1 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
DE0 2 co2 stored,DE0 2 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
DE0 3 co2 stored,DE0 3 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NO1 0 battery cluster,NO1 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
SE1 0 H2 Store cluster,SE1 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,2224.975719,0.0,True,0,inf,0.0,NaN
SE1 0 battery cluster,SE1 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN


In [23]:
n.links.loc[n.links["bus0"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 OCGT methanol,EU methanol,DE0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 1 OCGT methanol,EU methanol,DE0 1,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 2 OCGT methanol,EU methanol,DE0 2,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 3 OCGT methanol,EU methanol,DE0 3,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 4 OCGT methanol,EU methanol,DE0 4,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 5 OCGT methanol,EU methanol,DE0 5,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 6 OCGT methanol,EU methanol,DE0 6,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 7 OCGT methanol,EU methanol,DE0 7,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DK0 0 OCGT methanol,EU methanol,DK0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0


In [24]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
name,,,,,
DC,0.0,#8a1caf,DC,inf,0.0
AC,0.0,#70af1d,AC,inf,0.0
nuclear,0.0,#ff8c00,nuclear,inf,0.0
offwind-dc,0.0,#74c6f2,Offshore Wind (DC),inf,0.0
offwind-ac,0.0,#6895dd,Offshore Wind (AC),inf,0.0
...,...,...,...,...,...
rural air heat pump,0.0,#36eb41,rural air heat pump,inf,0.0
urban decentral resistive heater,0.0,#d8f9b8,urban decentral resistive heater,inf,0.0
urban central solid biomass CHP CC,0.0,#6c5d28,urban central solid biomass CHP CC,inf,0.0


In [25]:
n.global_constraints

,type,investment_period,bus,carrier_attribute,sense,constant,mu
name,,,,,,,
lv_limit,transmission_volume_expansion_limit,NaN,,"AC, DC",<=,7.078807e+07,0.0
biomass limit,operational_limit,NaN,,solid biomass,<=,3.278577e+08,0.0
CO2Limit,co2_atmosphere,NaN,,co2_emissions,<=,0.000000e+00,0.0


In [26]:
n.loads

,bus,carrier,type,p_set,q_set,sign,active
name,,,,,,,
DE0 0,DE0 0 low voltage,electricity,,0.0,0.0,-1.0,True
DE0 1,DE0 1 low voltage,electricity,,0.0,0.0,-1.0,True
DE0 2,DE0 2 low voltage,electricity,,0.0,0.0,-1.0,True
DE0 3,DE0 3 low voltage,electricity,,0.0,0.0,-1.0,True
DE0 4,DE0 4 low voltage,electricity,,0.0,0.0,-1.0,True
...,...,...,...,...,...,...,...
NO1 0 urban decentral heat,NO1 0 urban decentral heat,urban decentral heat,,0.0,0.0,-1.0,True
SE1 0 rural heat,SE1 0 rural heat,rural heat,,0.0,0.0,-1.0,True
SE1 0 urban decentral heat,SE1 0 urban decentral heat,urban decentral heat,,0.0,0.0,-1.0,True


In [27]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
relation/13295785-515-DC,NO1 0,GB2 0,,DC,0.955943,True,0,inf,1400.0,0.0,...,0.0,relation/13295785,LINESTRING (-1.5404269162550226 55.14647596191...,1.0,0.983350,,NaN,,False,1068.160241
relation/14126301-450-DC,GB2 1,NL0 0,,DC,0.971354,True,0,inf,1000.0,0.0,...,0.0,relation/14126301,LINESTRING (0.7161575436002887 51.440498299145...,1.0,0.977109,,NaN,,False,380.828296
relation/15775538-600-DC,GB2 2,GB2 4,,DC,0.971389,True,0,inf,2250.0,0.0,...,0.0,relation/15775538,LINESTRING (-4.894821189854914 55.718036768358...,1.0,0.892448,,NaN,,False,379.300168
relation/15781671-525-DC,GB2 0,DK0 0,,DC,0.963113,True,0,inf,1400.0,0.0,...,0.0,relation/15781671,LINESTRING (-0.2365005345670103 52.92099253311...,1.0,0.816407,,NaN,,False,746.986775
relation/16213216-525-DC,DE0 5,NO1 0,,DC,0.959162,True,0,inf,1400.0,0.0,...,0.0,relation/16213216,LINESTRING (6.7544309946114405 58.669060406684...,1.0,0.802530,,NaN,,False,923.664577
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NO1 0 battery discharger cluster,NO1 0 battery cluster,NO1 0 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN
SE1 0 battery charger cluster,SE1 0 cluster,SE1 0 battery cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 0 battery discharger cluster,SE1 0 battery cluster,SE1 0 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN


In [28]:
n.links.loc[n.links.index.str.contains("cluster"),n.links.columns.str.contains("bus")]

,bus0,bus1,bus4,bus3,bus2
name,,,,,
DE0 0 H2 Electrolysis cluster,DE0 0 cluster,DE0 0 H2 cluster,,,
DE0 0 methanolisation cluster,DE0 0 H2 cluster,DE0 0 methanol cluster,DE0 0 urban central heat,DE0 0 co2 stored,DE0 0 cluster
DE0 0 methanol cluster,DE0 0 methanol cluster,EU methanol,,,
DE0 1 H2 Electrolysis cluster,DE0 1 cluster,DE0 1 H2 cluster,,,
DE0 1 methanolisation cluster,DE0 1 H2 cluster,DE0 1 methanol cluster,DE0 1 urban central heat,DE0 1 co2 stored,DE0 1 cluster
...,...,...,...,...,...
NO1 0 battery discharger cluster,NO1 0 battery cluster,NO1 0 cluster,,,
SE1 0 battery charger cluster,SE1 0 cluster,SE1 0 battery cluster,,,
SE1 0 battery discharger cluster,SE1 0 battery cluster,SE1 0 cluster,,,


In [29]:
print(n.generators.loc[n.generators.index.str.contains("solar")&
    ~n.generators.index.str.contains("solar-hsat") &~n.generators.index.str.contains("solar thermal") &~n.generators.index.str.contains("solar rooftop")])

                                 bus control type        p_nom  p_nom_mod  \
name                                                                        
DE0 0 0 solar                  DE0 0      PQ         98.802080        0.0   
DE0 0 1 solar                  DE0 0      PQ       3665.427299        0.0   
DE0 0 2 solar                  DE0 0      PQ       6005.760077        0.0   
DE0 0 3 solar                  DE0 0      PQ       2935.844711        0.0   
DE0 0 4 solar                  DE0 0      PQ         30.153724        0.0   
...                              ...     ...  ...          ...        ...   
GB3 0 4 solar cluster  GB3 0 cluster      PQ          0.000000        0.0   
NL0 0 4 solar cluster  NL0 0 cluster      PQ          0.000000        0.0   
NO1 0 4 solar cluster  NO1 0 cluster      PQ          0.000000        0.0   
SE1 0 4 solar cluster  SE1 0 cluster      PQ          0.000000        0.0   
SE1 1 4 solar cluster  SE1 1 cluster      PQ          0.000000        0.0   

In [30]:
print(n.generators.loc[n.generators.index.str.contains("solar cluster"), n.generators.columns.isin(['p_nom_max','carrier','location','pnom_extendable'])])


                         p_nom_max carrier location
name                                               
DE0 0 4 solar cluster  1000.000000   solar         
DE0 1 4 solar cluster  1000.000000   solar         
DE0 2 4 solar cluster   269.893478   solar         
DE0 2 3 solar cluster   730.106522   solar         
DE0 3 4 solar cluster  1000.000000   solar         
DE0 4 4 solar cluster  1000.000000   solar         
DE0 5 4 solar cluster   597.086551   solar         
DE0 5 3 solar cluster   402.913449   solar         
DE0 6 4 solar cluster   129.703082   solar         
DE0 6 3 solar cluster   870.296918   solar         
DE0 7 4 solar cluster  1000.000000   solar         
DK0 0 4 solar cluster   165.931050   solar         
DK0 0 3 solar cluster   834.068950   solar         
DK1 0 4 solar cluster   971.684185   solar         
DK1 0 3 solar cluster    28.315815   solar         
GB2 0 4 solar cluster  1000.000000   solar         
GB2 1 4 solar cluster  1000.000000   solar         
GB2 2 4 sola

In [31]:
n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.str.contains("cluster")]

name,DE0 0 4 solar cluster,DE0 0 0 solar-hsat cluster,DE0 0 4 onwind cluster,DE0 0 3 onwind cluster,DE0 1 4 solar cluster,DE0 1 0 solar-hsat cluster,DE0 1 4 onwind cluster,DE0 2 4 solar cluster,DE0 2 3 solar cluster,DE0 2 0 solar-hsat cluster,...,NL0 0 3 onwind cluster,NO1 0 4 solar cluster,NO1 0 0 solar-hsat cluster,NO1 0 4 onwind cluster,SE1 0 4 solar cluster,SE1 0 0 solar-hsat cluster,SE1 0 4 onwind cluster,SE1 1 4 solar cluster,SE1 1 0 solar-hsat cluster,SE1 1 4 onwind cluster
snapshot,,,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.009373,0.041229,0.999350,0.985263,0.038094,0.026673,0.795164,0.113277,0.156725,0.101547,...,0.577368,0.013872,0.009266,0.617372,0.007216,0.027491,0.992861,0.026330,0.019140,0.434222
2013-01-01 12:00:00,0.003669,0.012184,0.626917,0.491829,0.015363,0.012344,0.321672,0.066025,0.099226,0.050599,...,0.851731,0.007533,0.004823,0.403871,0.002338,0.008017,0.541646,0.003967,0.003081,0.179009
2013-01-02 00:00:00,0.018285,0.055901,0.942058,0.802889,0.117064,0.068300,0.173261,0.088623,0.074397,0.107956,...,0.743640,0.012977,0.008648,0.238663,0.021945,0.031705,0.666617,0.020717,0.013949,0.185520
2013-01-02 12:00:00,0.020800,0.016345,0.992998,0.903323,0.066501,0.033015,0.169112,0.047009,0.052284,0.047092,...,0.799629,0.006878,0.004207,0.408567,0.004903,0.014160,0.782770,0.003123,0.002344,0.200599
2013-01-03 00:00:00,0.027693,0.016241,0.999435,0.976978,0.031356,0.018432,0.464308,0.038478,0.043784,0.026644,...,0.973061,0.009181,0.005952,0.688372,0.010255,0.011225,0.834281,0.007675,0.005723,0.091640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-29 12:00:00,0.003591,0.020665,0.951144,0.774361,0.041248,0.027588,0.158013,0.014305,0.016217,0.035080,...,0.886826,0.004667,0.001965,0.161766,0.005210,0.007175,0.900232,0.006373,0.004764,0.824024
2013-12-30 00:00:00,0.100585,0.119247,0.947525,0.759803,0.110926,0.094557,0.100908,0.131453,0.164501,0.141464,...,0.975430,0.007159,0.004548,0.487115,0.069811,0.033784,0.967619,0.055146,0.037400,0.636971
2013-12-30 12:00:00,0.034378,0.061177,0.890941,0.761331,0.088752,0.074564,0.215838,0.103312,0.110259,0.081787,...,0.998983,0.002431,0.001642,0.981489,0.016077,0.011450,0.872472,0.008453,0.006359,0.538245


**Exporting**

In [32]:
n.export_to_netcdf(fn)


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'resources/Nordics20_test/networks/base_s_20__12h_2050.nc contains: lines, buses, global_constraints, sub_networks, generators, storage_units, loads, carriers, links, stores


<xarray.Dataset> Size: 5MB
Dimensions:                               (snapshots: 730, lines_i: 33,
                                           buses_i: 588,
                                           global_constraints_i: 3,
                                           sub_networks_i: 4,
                                           generators_i: 646,
                                           generators_t_p_max_pu_i: 497,
                                           ...
                                           carriers_i: 120, links_i: 1667,
                                           links_t_efficiency_i: 80,
                                           links_t_p_max_pu_i: 40,
                                           stores_i: 284,
                                           stores_t_e_min_pu_i: 20,
                                           stores_t_e_max_pu_i: 40)
Coordinates: (12/18)
  * snapshots                             (snapshots) int64 6kB 0 1 ... 728 729
  * lines_i                               (lines_i) object 264B '0' '1' ... '9'
  * buses_i                               (buses_i) object 5kB 'DE0 0' ... 'S...
  * global_constraints_i                  (global_constraints_i) object 24B '...
  * sub_networks_i                        (sub_networks_i) object 32B '0' ......
  * generators_i                          (generators_i) object 5kB 'GB2 0 nu...
    ...                                    ...
  * links_i                               (links_i) object 13kB 'relation/132...
  * links_t_efficiency_i                  (links_t_efficiency_i) object 640B ...
  * links_t_p_max_pu_i                    (links_t_p_max_pu_i) object 320B 'D...
  * stores_i                              (stores_i) object 2kB 'co2 atmosphe...
  * stores_t_e_min_pu_i                   (stores_t_e_min_pu_i) object 160B '...
  * stores_t_e_max_pu_i                   (stores_t_e_max_pu_i) object 320B '...
Data variables: (12/127)
    snapshots_snapshot                    (snapshots) datetime64[ns] 6kB 2013...
    snapshots_objective                   (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_stores                      (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_generators                  (snapshots) float64 6kB 12.0 ... 12.0
    lines_bus0                            (lines_i) object 264B 'DE0 0' ... '...
    lines_bus1                            (lines_i) object 264B 'DE0 5' ... '...
    ...                                    ...
    stores_capital_cost                   (stores_i) float64 2kB 0.0 ... 6.42...
    stores_standing_loss                  (stores_i) float64 2kB 0.0 0.0 ... 0.0
    stores_lifetime                       (stores_i) float64 2kB inf inf ... inf
    stores_location                       (stores_i) object 2kB '' '' ... nan
    stores_t_e_min_pu                     (snapshots, stores_t_e_min_pu_i) float64 117kB ...
    stores_t_e_max_pu                     (snapshots, stores_t_e_max_pu_i) float64 234kB ...
Attributes:
    network_name:           Unnamed Network
    network_pypsa_version:  1.0.6
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"version": "v2025.07.0", "tutorial": false, "log...